# Session 2, Module 03: Dictionaries Deep Dive


This module covers:
- Creation methods: literal, dict(), dict comprehension, fromkeys()
- Access: [], .get(), .setdefault()
- Methods: keys(), values(), items(), update(), pop(), popitem()
- Merging dicts: {**d1, **d2} and d1 | d2 (Python 3.9+)
- Nested dicts and safe access patterns
- Performance: O(1) lookup

Data Engineering Context:
Dictionaries are essential for lookup tables, configuration management,
JSON handling, and mapping source to target schemas.


## Dictionary Creation


In [1]:
print("=== Dictionary Creation ===")

# Literal syntax (most common)
config = {
    "host": "localhost",
    "port": 5432,
    "database": "warehouse",
}
print(f"Literal: {config}")

# dict() constructor
config2 = dict(host="localhost", port=5432, database="warehouse")
print(f"dict(): {config2}")

# From list of tuples
pairs = [("a", 1), ("b", 2), ("c", 3)]
from_pairs = dict(pairs)
print(f"From pairs: {from_pairs}")

# Dict comprehension
columns = ["id", "name", "email"]
column_index = {col: idx for idx, col in enumerate(columns)}
print(f"Comprehension: {column_index}")

# fromkeys() - Create dict with default value
keys = ["status", "created_at", "updated_at"]
defaults = dict.fromkeys(keys, None)
print(f"fromkeys: {defaults}")

# Empty dict
empty = {}
also_empty = dict()

=== Dictionary Creation ===
Literal: {'host': 'localhost', 'port': 5432, 'database': 'warehouse'}
dict(): {'host': 'localhost', 'port': 5432, 'database': 'warehouse'}
From pairs: {'a': 1, 'b': 2, 'c': 3}
Comprehension: {'id': 0, 'name': 1, 'email': 2}
fromkeys: {'status': None, 'created_at': None, 'updated_at': None}


## Accessing Values


In [2]:
print("\n=== Accessing Values ===")

config = {
    "host": "localhost",
    "port": 5432,
    "database": "warehouse",
    "timeout": None,
}

# Direct access with []
print(f"config['host']: {config['host']}")


=== Accessing Values ===
config['host']: localhost


KeyError if key doesn't exist
print(config['password'])  # KeyError!
.get() - Safe access with default

In [3]:
print(f"config.get('host'): {config.get('host')}")
print(f"config.get('password'): {config.get('password')}")  # None
print(f"config.get('password', 'default'): {config.get('password', 'default')}")

# IMPORTANT: .get() returns None even if key exists with None value
print(f"\nconfig.get('timeout'): {config.get('timeout')}")  # None (exists)
print(f"config.get('missing'): {config.get('missing')}")    # None (doesn't exist)

# Check if key exists
if "timeout" in config:
    print(f"timeout exists: {config['timeout']}")

# .setdefault() - Get value or set default if missing
config.setdefault("max_connections", 10)
print(f"\nAfter setdefault: {config}")
# If key existed, it's not modified
config.setdefault("port", 9999)  # port stays 5432
print(f"port unchanged: {config['port']}")

config.get('host'): localhost
config.get('password'): None
config.get('password', 'default'): default

config.get('timeout'): None
config.get('missing'): None
timeout exists: None

After setdefault: {'host': 'localhost', 'port': 5432, 'database': 'warehouse', 'timeout': None, 'max_connections': 10}
port unchanged: 5432


## Dictionary Methods


In [4]:
print("\n=== Dictionary Methods ===")

data = {"a": 1, "b": 2, "c": 3}

# keys(), values(), items() - Return view objects
print(f"keys():   {list(data.keys())}")
print(f"values(): {list(data.values())}")
print(f"items():  {list(data.items())}")

# Iterate over keys (default)
print("\nIterating over dict:")
for key in data:
    print(f"  {key}: {data[key]}")

# Iterate over key-value pairs
print("\nIterating with items():")
for key, value in data.items():
    print(f"  {key}: {value}")

# update() - Add or update multiple key-value pairs
config = {"host": "localhost", "port": 5432}
config.update({"port": 5433, "database": "prod"})  # Updates port, adds database
print(f"\nAfter update(): {config}")

# Can also use keyword arguments
config.update(timeout=30, ssl=True)
print(f"After update() with kwargs: {config}")

# pop() - Remove and return value
port = config.pop("port")
print(f"\npop('port') returned: {port}")
print(f"config after pop: {config}")

# pop() with default (no KeyError if missing)
missing = config.pop("nonexistent", "default_value")
print(f"pop('nonexistent', 'default_value'): {missing}")

# popitem() - Remove and return last inserted item (LIFO)
config = {"a": 1, "b": 2, "c": 3}
last = config.popitem()
print(f"\npopitem() returned: {last}")
print(f"config after popitem: {config}")

# clear() - Remove all items
config.clear()
print(f"After clear(): {config}")


=== Dictionary Methods ===
keys():   ['a', 'b', 'c']
values(): [1, 2, 3]
items():  [('a', 1), ('b', 2), ('c', 3)]

Iterating over dict:
  a: 1
  b: 2
  c: 3

Iterating with items():
  a: 1
  b: 2
  c: 3

After update(): {'host': 'localhost', 'port': 5433, 'database': 'prod'}
After update() with kwargs: {'host': 'localhost', 'port': 5433, 'database': 'prod', 'timeout': 30, 'ssl': True}

pop('port') returned: 5433
config after pop: {'host': 'localhost', 'database': 'prod', 'timeout': 30, 'ssl': True}
pop('nonexistent', 'default_value'): default_value

popitem() returned: ('c', 3)
config after popitem: {'a': 1, 'b': 2}
After clear(): {}


## Merging Dictionaries


In [5]:
print("\n=== Merging Dictionaries ===")

defaults = {"timeout": 30, "retries": 3, "debug": False}
user_config = {"timeout": 60, "custom": True}

# Method 1: ** unpacking (creates new dict)
merged = {**defaults, **user_config}  # user_config values override
print(f"Merged with **: {merged}")

# Method 2: | operator (Python 3.9+)
merged = defaults | user_config
print(f"Merged with |: {merged}")

# Method 3: |= for in-place merge (Python 3.9+)
config = {"a": 1, "b": 2}
config |= {"b": 3, "c": 4}
print(f"After |=: {config}")

# Method 4: update() (modifies in place)
config = {"a": 1, "b": 2}
config.update({"b": 3, "c": 4})
print(f"After update(): {config}")

# Merging multiple dicts
d1 = {"a": 1}
d2 = {"b": 2}
d3 = {"c": 3}
merged = {**d1, **d2, **d3}
print(f"\nMerged 3 dicts: {merged}")


=== Merging Dictionaries ===
Merged with **: {'timeout': 60, 'retries': 3, 'debug': False, 'custom': True}
Merged with |: {'timeout': 60, 'retries': 3, 'debug': False, 'custom': True}
After |=: {'a': 1, 'b': 3, 'c': 4}
After update(): {'a': 1, 'b': 3, 'c': 4}

Merged 3 dicts: {'a': 1, 'b': 2, 'c': 3}


## Nested Dictionaries


In [6]:
print("\n=== Nested Dictionaries ===")

# Pipeline configuration with nested structure
pipeline_config = {
    "source": {
        "type": "database",
        "connection": {
            "host": "db.example.com",
            "port": 5432,
            "database": "source_db",
        },
        "query": "SELECT * FROM customers",
    },
    "destination": {
        "type": "file",
        "path": "/data/output/customers.parquet",
    },
    "transformations": ["clean", "dedupe", "validate"],
}

# Direct nested access
host = pipeline_config["source"]["connection"]["host"]
print(f"Source host: {host}")

# Chained .get() for safe access
port = pipeline_config.get("source", {}).get("connection", {}).get("port")
print(f"Source port: {port}")

# Safe access helper function
def get_nested(d: dict, *keys, default=None):
    """Safely get nested dictionary value."""
    for key in keys:
        if isinstance(d, dict):
            d = d.get(key)
        else:
            return default
    return d if d is not None else default


print(f"\nget_nested for host: {get_nested(pipeline_config, 'source', 'connection', 'host')}")
print(f"get_nested for missing: {get_nested(pipeline_config, 'source', 'missing', 'key')}")
print(f"get_nested with default: {get_nested(pipeline_config, 'missing', default='N/A')}")

# Modifying nested values
pipeline_config["source"]["connection"]["port"] = 5433
print(f"Modified port: {pipeline_config['source']['connection']['port']}")

# Adding new nested keys
pipeline_config["source"]["connection"]["ssl"] = True
print(f"Added ssl: {pipeline_config['source']['connection']}")


=== Nested Dictionaries ===
Source host: db.example.com
Source port: 5432

get_nested for host: db.example.com
get_nested for missing: None
get_nested with default: N/A
Modified port: 5433
Added ssl: {'host': 'db.example.com', 'port': 5433, 'database': 'source_db', 'ssl': True}


## Performance: O(1) Lookup


In [7]:
print("\n=== Performance ===")

print("""
Dictionary Operations (Big O):
┌──────────────────────┬─────────────┐
│ Operation            │ Time        │
├──────────────────────┼─────────────┤
│ Get item d[key]      │ O(1)        │
│ Set item d[key] = v  │ O(1)        │
│ Delete del d[key]    │ O(1)        │
│ Key in dict          │ O(1)        │
│ Get keys/values/items│ O(1)*       │
│ Iterate              │ O(n)        │
│ Copy                 │ O(n)        │
└──────────────────────┴─────────────┘
* View objects are O(1) to create, O(n) to convert to list

Why O(1)? Hash table implementation!
- Python hashes the key to find storage location
- Direct access without searching
- Keys must be hashable (immutable)
""")

# Compare dict lookup vs list search
import time

# Create lookup data
size = 100000
lookup_dict = {f"key_{i}": i for i in range(size)}
lookup_list = [(f"key_{i}", i) for i in range(size)]

# Dict lookup
search_key = f"key_{size - 1}"

start = time.perf_counter()
result = lookup_dict.get(search_key)
dict_time = time.perf_counter() - start

# List search (simulated)
start = time.perf_counter()
result = None
for k, v in lookup_list:
    if k == search_key:
        result = v
        break
list_time = time.perf_counter() - start

print(f"Dict lookup time: {dict_time:.6f}s")
print(f"List search time: {list_time:.6f}s")
print(f"Dict is ~{list_time/dict_time:.0f}x faster")


=== Performance ===

Dictionary Operations (Big O):
┌──────────────────────┬─────────────┐
│ Operation            │ Time        │
├──────────────────────┼─────────────┤
│ Get item d[key]      │ O(1)        │
│ Set item d[key] = v  │ O(1)        │
│ Delete del d[key]    │ O(1)        │
│ Key in dict          │ O(1)        │
│ Get keys/values/items│ O(1)*       │
│ Iterate              │ O(n)        │
│ Copy                 │ O(n)        │
└──────────────────────┴─────────────┘
* View objects are O(1) to create, O(n) to convert to list

Why O(1)? Hash table implementation!
- Python hashes the key to find storage location
- Direct access without searching
- Keys must be hashable (immutable)

Dict lookup time: 0.000033s
List search time: 0.005675s
Dict is ~172x faster


## Practical: Lookup Table And Mapping


In [8]:
print("\n=== Practical: Lookup Table and Mapping ===")

# Create column mapping for ETL
source_columns = ["cust_id", "cust_name", "cust_email", "created_dt"]
target_columns = ["customer_id", "name", "email", "created_at"]

column_mapping = dict(zip(source_columns, target_columns))
print(f"Column mapping: {column_mapping}")

# Transform a record using the mapping
source_record = {
    "cust_id": 1,
    "cust_name": "Alice",
    "cust_email": "alice@example.com",
    "created_dt": "2024-01-15",
}

target_record = {
    column_mapping.get(k, k): v  # Map column name, keep original if not mapped
    for k, v in source_record.items()
}
print(f"Transformed: {target_record}")

# Lookup table for status codes
status_lookup = {
    "A": "Active",
    "I": "Inactive",
    "P": "Pending",
    "D": "Deleted",
}

records = [
    {"id": 1, "status": "A"},
    {"id": 2, "status": "P"},
    {"id": 3, "status": "X"},  # Unknown status
]

for record in records:
    code = record["status"]
    status_name = status_lookup.get(code, f"Unknown ({code})")
    print(f"  ID {record['id']}: {status_name}")


=== Practical: Lookup Table and Mapping ===
Column mapping: {'cust_id': 'customer_id', 'cust_name': 'name', 'cust_email': 'email', 'created_dt': 'created_at'}
Transformed: {'customer_id': 1, 'name': 'Alice', 'email': 'alice@example.com', 'created_at': '2024-01-15'}
  ID 1: Active
  ID 2: Pending
  ID 3: Unknown (X)


## Summary


In [9]:
print("\n=== Summary ===")
print("""
Creation:
  {"k": v}         - Literal
  dict(k=v)        - Constructor
  dict(pairs)      - From list of tuples
  {k: v for x}     - Comprehension
  dict.fromkeys()  - Default values

Access:
  d["key"]         - Direct (KeyError if missing)
  d.get("key")     - Safe (None if missing)
  d.get("key", 0)  - Safe with default
  d.setdefault()   - Get or set default

Methods:
  .keys(), .values(), .items() - View objects
  .update(other)   - Add/update multiple
  .pop(key)        - Remove and return
  .popitem()       - Remove last item

Merging:
  {**d1, **d2}     - Unpack (new dict)
  d1 | d2          - Union operator (3.9+)
  d1.update(d2)    - In-place update

Performance:
  O(1) for get, set, delete, membership
  Use dict for lookups, not list!
""")


=== Summary ===

Creation:
  {"k": v}         - Literal
  dict(k=v)        - Constructor
  dict(pairs)      - From list of tuples
  {k: v for x}     - Comprehension
  dict.fromkeys()  - Default values

Access:
  d["key"]         - Direct (KeyError if missing)
  d.get("key")     - Safe (None if missing)
  d.get("key", 0)  - Safe with default
  d.setdefault()   - Get or set default

Methods:
  .keys(), .values(), .items() - View objects
  .update(other)   - Add/update multiple
  .pop(key)        - Remove and return
  .popitem()       - Remove last item

Merging:
  {**d1, **d2}     - Unpack (new dict)
  d1 | d2          - Union operator (3.9+)
  d1.update(d2)    - In-place update

Performance:
  O(1) for get, set, delete, membership
  Use dict for lookups, not list!

